# Rossmann Retail Demand Forecasting

## Problem Framing

### Problem Statement
Rossmann store managers need to know how much each store will sell in the coming days to plan staffing and stock. A forecast that is too low means lost sales. One that is too high means waste and tied-up cash.

### Task
Supervised regression: predict `Sales` (euros) for each store and day, for the next 7 days. One model with lag features at least 7 days back (lags 7, 14, 21, 28) is reused for each of the 7 forecast days.

### Metrics
- **WAPE** (total absolute error divided by total sales) is the headline metric.
- **MAE** in euros is reported alongside it.
- A model must beat two baselines: sales on the same weekday last week, and the store's average for that weekday.
- Closed days (`Open = 0`) are dropped from training and scoring. Their `Sales` is always 0 because of the store schedule, not demand, and they would pull MAE down. At prediction time, a closed store gets a forecast of 0.



### Validation plan
- Three rolling backtest folds, each trained only on earlier data, are used to compare and tune models.
- The last 6 weeks of `train.csv` are a final holdout, scored once. (Kaggle's `test.csv` has no `Sales`, so it isn't used.)
- Only things known before the day starts are used for the forecast days: `Open` (store schedule), `Promo`, `StateHoliday` and `SchoolHoliday` are all planned in advance. Same-day `Customers` is not, so it is not used as a feature. That would be data leakage.


## Setup

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# Constants
HORIZON = 7          # forecast days ahead
HOLDOUT_DAYS = 42    # final holdout: the last 6 weeks of train.csv
RANDOM_SEED = 1      # reproducible randomness

# Display settings
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Data Preparation

### Load data

- `train.csv` has one row per store per day.
- `store.csv` has one row per store. 
- `StateHoliday` is read as text because it mixes numbers and letters.

In [2]:
train = pd.read_csv(
    "../data/raw/train.csv",
    dtype={"StateHoliday": "str"},
    parse_dates=["Date"],
)

print(f"train: {train.shape[0]:,} rows x {train.shape[1]} columns")

train: 1,017,209 rows x 9 columns


In [3]:
store = pd.read_csv("../data/raw/store.csv")

print(f"store: {store.shape[0]:,} rows x {store.shape[1]} columns")

store: 1,115 rows x 10 columns


### Data Profiling

Checking the raw data before changing anything.

- Structure and types
- Date range
- Grain (duplicates) check
- Completeness check
- Missing data, range and value checks
- Target (`Sales`)
- Findings

#### Structure and types

Preview first few rows and check each column's info (non-null counts, data types)

In [4]:
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [5]:
train.tail()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
1017204,1111,2,2013-01-01,0,0,0,0,a,1
1017205,1112,2,2013-01-01,0,0,0,0,a,1
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1
1017208,1115,2,2013-01-01,0,0,0,0,a,1


The rows are sorted newest first (2015-07-31 down to 2013-01-01), and within each date by `Store`. Lags and rolling features need oldest-to-newest order, so cleaning sorts by `Store` and `Date`.

In [6]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[us]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  str           
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 69.8 MB


**Data Dictionary**

`train.csv` (one row per store per day)

| Column | Meaning |
|---|---|
| `Store` | Store ID, 1 to 1,115 |
| `Date` | Calendar day |
| `DayOfWeek` | 1 = Monday to 7 = Sunday |
| `Sales` | Turnover in euros that day (**target**) |
| `Customers` | Number of customers that day |
| `Open` | 0 = closed, 1 = open |
| `Promo` | Whether the store ran a promotion that day |
| `StateHoliday` | Public holiday type: `0` none, `a` public, `b` Easter, `c` Christmas |
| `SchoolHoliday` | Whether the store was affected by school closures |

In [7]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [8]:
store.info()

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 87.2 KB


**Data Dictionary**

`store.csv` (one row per store)

| Column | Meaning |
|---|---|
| `StoreType` | Store model, `a` to `d` |
| `Assortment` | Product range level: `a` basic, `b` extra, `c` extended |
| `CompetitionDistance` | Distance in metres to the nearest competitor |
| `CompetitionOpenSinceMonth`, `CompetitionOpenSinceYear` | When the nearest competitor opened |
| `Promo2` | Whether the store takes part in a continuing promotion (0 or 1) |
| `Promo2SinceWeek`, `Promo2SinceYear` | When the store joined that promotion |
| `PromoInterval` | Months when a Promo2 round restarts, e.g. `Feb,May,Aug,Nov` |

#### Date range

In [9]:
earliest_date = train["Date"].min()
latest_date = train["Date"].max()
days = (latest_date - earliest_date).days + 1
years = days / 365

print(f"Earliest date: {earliest_date.date()}")
print(f"Latest date: {latest_date.date()}")
print(f"Number of days: {days}")
print(f"Number of years: {years:.2f}")

Earliest date: 2013-01-01
Latest date: 2015-07-31
Number of days: 942
Number of years: 2.58


#### Grain check

Used to spot potential duplicates in our data

In [10]:
print(f"Number of duplicate rows: {train.duplicated(['Store', 'Date']).sum()}")

Number of duplicate rows: 0


#### Completeness check

Every store should have one row for every day, so 1,115 stores × 942 days. Missing rows matter because lags are built from dates, and a gap breaks any "7 rows back" shortcut.

In [11]:
n_stores = train["Store"].nunique()
expected_rows = n_stores * days
missing_rows = expected_rows - len(train)
missing_per_store = days - train.groupby("Store").size()

print(f"Expected rows (stores x days): {expected_rows:,}")
print(f"Actual rows: {len(train):,}")
print(f"Missing store-days: {missing_rows:,} ({missing_rows / expected_rows:.1%})")
print(f"Stores with gaps: {(missing_per_store > 0).sum()} of {n_stores}")
print(f"Worst store is missing {missing_per_store.max()} days")

Expected rows (stores x days): 1,050,330
Actual rows: 1,017,209
Missing store-days: 33,121 (3.2%)
Stores with gaps: 181 of 1115
Worst store is missing 184 days


33,121 store-days (about 3%) have no row at all. 181 stores have gaps, and the worst is missing 184 days. These are likely temporary closures, which EDA will check. Lags and rolling features must be built by calendar date, never by row position.

#### Missing data, range and value checks

**Train**

In [12]:
train.isnull().sum()

Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

In [13]:
train[["Sales", "Customers"]].describe()

,Sales,Customers
count,1.017209e+06,1.017209e+06
mean,5.773819e+03,6.331459e+02
std,3.849926e+03,4.644117e+02
min,0.000000e+00,0.000000e+00
25%,3.727000e+03,4.050000e+02
50%,5.744000e+03,6.090000e+02
75%,7.856000e+03,8.370000e+02
max,4.155100e+04,7.388000e+03


In [14]:
for col in ["DayOfWeek", "Open", "Promo", "StateHoliday", "SchoolHoliday"]:
    print(train[col].value_counts(dropna=False).sort_index(), "\n")

DayOfWeek
1    144730
2    145664
3    145665
4    145845
5    145845
6    144730
7    144730
Name: count, dtype: int64 

Open
0    172817
1    844392
Name: count, dtype: int64 

Promo
0    629129
1    388080
Name: count, dtype: int64 

StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64 

SchoolHoliday
0    835488
1    181721
Name: count, dtype: int64 



**train:** no missing values, and all categorical columns contain only expected values (flags are 0/1, `StateHoliday` is `0`/`a`/`b`/`c`). `Sales` and `Customers` have no negatives, but both have a minimum of 0, likely the 172,817 closed days (`Open = 0`). The Target section checks this. Both are also right-skewed (max `Sales` 41,551 vs mean about 5,774), which EDA will examine. `StateHoliday` stores "no holiday" as `0` next to the letter codes, so it needs recoding in cleaning.

**Store**

In [15]:
store.isnull().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

In [16]:
store.describe()

,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear
count,1115.00000,1112.000000,761.000000,761.000000,1115.000000,571.000000,571.000000
mean,558.00000,5404.901079,7.224704,2008.668857,0.512108,23.595447,2011.763573
std,322.01708,7663.174720,3.212348,6.195983,0.500078,14.141984,1.674935
min,1.00000,20.000000,1.000000,1900.000000,0.000000,1.000000,2009.000000
25%,279.50000,717.500000,4.000000,2006.000000,0.000000,13.000000,2011.000000
50%,558.00000,2325.000000,8.000000,2010.000000,1.000000,22.000000,2012.000000
75%,836.50000,6882.500000,10.000000,2013.000000,1.000000,37.000000,2013.000000
max,1115.00000,75860.000000,12.000000,2015.000000,1.000000,50.000000,2015.000000


In [17]:
store["CompetitionOpenSinceYear"].nsmallest(3)

814    1900.0
145    1961.0
328    1990.0
Name: CompetitionOpenSinceYear, dtype: float64

In [18]:
for col in ["StoreType", "Assortment", "PromoInterval"]:
    print(store[col].value_counts(dropna=False).sort_index(), "\n")

StoreType
a    602
b     17
c    148
d    348
Name: count, dtype: int64 

Assortment
a    593
b      9
c    513
Name: count, dtype: int64 

PromoInterval
Feb,May,Aug,Nov     130
Jan,Apr,Jul,Oct     335
Mar,Jun,Sept,Dec    106
NaN                 544
Name: count, dtype: int64 



**store:** the missing data has clear causes.

- `Promo2SinceWeek`, `Promo2SinceYear` and `PromoInterval` are missing for 544 stores, which matches the 544 stores with `Promo2 = 0`. These stores aren't in the Promo2 programme, so the values are structurally absent, not unknown.
- `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` are missing for 354 stores (32%), and `CompetitionDistance` for 3 stores. How to fill them is decided in cleaning.

Ranges are plausible except `CompetitionOpenSinceYear`, where one store has 1900, far below the next lowest year (1961). It is treated as an outlier and handled in cleaning.

`CompetitionDistance` is right-skewed (median about 2,325 m, max 75,860 m). Categories are consistent: `StoreType` (a to d), `Assortment` (a to c) and three `PromoInterval` patterns. `PromoInterval` spells September as "Sept", which matters when parsing months.

#### Target (`Sales`)

Is `Sales = 0` the same as a closed store (`Open = 0`)? Zero sales on an open day would be suspicious.

In [19]:
pd.crosstab(
    train["Sales"].eq(0).map({True: "Sales = 0", False: "Sales > 0"}),
    train["Open"].map({0: "Closed", 1: "Open"}),
    rownames=["Sales"],
    colnames=["Store status"],
)

Store status,Closed,Open
Sales,,
Sales = 0,172817,54
Sales > 0,0,844338


#### Findings

- **Row order:** `train` is sorted newest first. Cleaning sorts by `Store` and `Date` before any lag or rolling feature is built.
- **Grain:** one row per store per day, no duplicates. `train` covers 1,115 stores from 2013-01-01 to 2015-07-31.
- **Completeness:** 33,121 of 1,050,330 store-days (about 3%) have no row. 181 stores have gaps, the worst missing 184 days. Lags must be built by calendar date, not row position.
- **Missing data:** `train` has none. In `store`, the Promo2 fields are missing for the 544 stores not in Promo2 (structural). The competition fields are missing for 354 stores (unknown), and `CompetitionDistance` for 3.
- **Target:** closed days always have `Sales = 0`. 54 open days also have zero sales, so cleaning decides what to do with them.
- **Suspicious values:** one store has `CompetitionOpenSinceYear = 1900`, far below the next lowest year (1961). It is an outlier, handled in cleaning.
- **Formatting issues:** `StateHoliday` mixes `0` with letter codes, and `PromoInterval` spells September as "Sept".
- **Skew:** `Sales`, `Customers` and `CompetitionDistance` are right-skewed, to be examined in EDA.

### Clean the data

## Exploratory Data Analysis